# Household energy prediction - daily model
Forecasting 5 | ICT30016

Rebuilds the daily table from the raw UCI file, then trains the baseline + ML models.
Upload `individualhouseholdelectricpowerconsumption.zip` when prompted in cell 2.

In [ ]:
!pip install -q xgboost

In [ ]:
import os, json, hashlib, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
from google.colab import files
up = files.upload()

with zipfile.ZipFile(next(iter(up))) as z:
    z.extractall("raw")

RAW = "raw/household_power_consumption.txt"
os.path.getsize(RAW) / 1e6

In [ ]:
# check the format before parsing - it's ; separated and uses ? for missing
!head -3 raw/household_power_consumption.txt

### Load raw

In [ ]:
cols = ["Global_active_power","Global_reactive_power","Voltage","Global_intensity",
        "Sub_metering_1","Sub_metering_2","Sub_metering_3"]

df = pd.read_csv(RAW, sep=";", na_values=["?"], low_memory=False, dtype=str)

# dates are dd/mm/yyyy - have to say so or pandas guesses wrong
df["timestamp"] = pd.to_datetime(df.Date + " " + df.Time, format="%d/%m/%Y %H:%M:%S")
for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.drop(columns=["Date","Time"]).sort_values("timestamp")

df.shape

In [ ]:
df.head()

In [ ]:
# how bad is the missing data
print(df[cols].isna().all(axis=1).sum(), "rows fully blank")
print(df.timestamp.min(), "->", df.timestamp.max())

# are any minutes actually absent, or just blank?
exp = int((df.timestamp.max() - df.timestamp.min()).total_seconds() // 60) + 1
print("expected", exp, "got", len(df), "-> gaps:", exp - len(df))

Rows are all there, ~1.25% just have no values. So a coverage rule per day will work.

In [ ]:
g = df.set_index("timestamp").resample("D")

daily = pd.DataFrame({
    # Global_active_power is kW every minute -> sum is kW-min, /60 for kWh
    "kwh": g.Global_active_power.sum(min_count=1) / 60,
    "reactive_kvarh": g.Global_reactive_power.sum(min_count=1) / 60,
    "voltage": g.Voltage.mean(),
    "intensity": g.Global_intensity.mean(),
    "sub1_wh": g.Sub_metering_1.sum(min_count=1),
    "sub2_wh": g.Sub_metering_2.sum(min_count=1),
    "sub3_wh": g.Sub_metering_3.sum(min_count=1),
    "mins": g.Global_active_power.count(),
})
daily["other_wh"] = daily.kwh*1000 - daily.sub1_wh - daily.sub2_wh - daily.sub3_wh

daily.mins.describe()

In [ ]:
# drop days that aren't nearly complete, otherwise a 3-hour day
# looks like a very low consumption day
full = daily[daily.mins >= 0.9*1440].drop(columns="mins")
full.index.name = "date"
full = full.reset_index()

print(len(daily), "calendar days ->", len(full), "kept")
full.to_csv("data/household_daily.csv", index=False)
full.head()

### Sanity checks

In [ ]:
print("rows      ", len(full))
print("range     ", full.date.min().date(), "->", full.date.max().date())
print("mean kwh  ", round(full.kwh.mean(), 2))
print("min/max   ", round(full.kwh.min(),2), round(full.kwh.max(),2))
print("nulls     ", full.kwh.isna().sum())

26 kWh/day is realistic for a house, so the /60 is right. (Without it you'd get ~1571.)

In [ ]:
# missing day blocks after the coverage filter
gap = full.date.diff().dt.days
full.loc[gap[gap>1].index-1].assign(next_date=full.date[gap[gap>1].index].values)[["date","next_date"]]

In [ ]:
full.plot(x="date", y="kwh", figsize=(13,3), lw=.6, legend=False)
plt.title("daily kWh"); plt.show()

In [ ]:
# provenance so the result can be traced back to this exact file
sha = hashlib.sha256(open(RAW,"rb").read()).hexdigest()
json.dump({
    "source": "UCI individual household electric power consumption",
    "url": "https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption",
    "raw_sha256": sha,
    "raw_rows": len(df),
    "daily_rows": len(full),
    "from": str(full.date.min().date()),
    "to": str(full.date.max().date()),
    "coverage_rule": 0.9,
    "seed": SEED,
}, open("data/manifest.json","w"), indent=2)
sha[:16]

### Features

In [ ]:
d = pd.read_csv("data/household_daily.csv", parse_dates=["date"])
lagme = ["reactive_kvarh","voltage","intensity","sub1_wh","sub2_wh","sub3_wh","other_wh"]

X = d[["date","kwh"]].copy()

for l in [1,2,3,7,14]:
    X[f"lag_{l}"] = d.kwh.shift(l)

for w in [7,14,28]:
    X[f"rollmean_{w}"] = d.kwh.shift(1).rolling(w).mean()
    X[f"rollstd_{w}"]  = d.kwh.shift(1).rolling(w).std()

# these are only known at the END of the day, so shift them too
for c in lagme:
    X[f"{c}_l1"] = d[c].shift(1)
    X[f"{c}_l7"] = d[c].shift(7)

t = X.date.dt
X["dow"] = t.dayofweek
X["month"] = t.month
X["doy"] = t.dayofyear
X["weekend"] = (t.dayofweek >= 5).astype(int)
X["dow_sin"], X["dow_cos"] = np.sin(2*np.pi*X.dow/7), np.cos(2*np.pi*X.dow/7)
X["doy_sin"], X["doy_cos"] = np.sin(2*np.pi*X.doy/365.25), np.cos(2*np.pi*X.doy/365.25)

X = X.dropna().reset_index(drop=True)
X.shape

In [ ]:
# leakage check - lag_1 should equal yesterday's kwh, nothing else
y_prev = d[["date","kwh"]].copy()
y_prev["date"] += pd.Timedelta(days=1)
chk = X[["date","lag_1"]].merge(y_prev, on="date")
assert (chk.lag_1 - chk.kwh).abs().max() < 1e-9
assert not set(lagme) & set(X.columns)
print("ok")

### Split + models

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# time order, not random
n = len(X)
te = X.iloc[int(n*0.8):]
tv = X.iloc[:int(n*0.8)]
va = tv.iloc[int(len(tv)*0.8):]
tr = tv.iloc[:int(len(tv)*0.8)]

drop = ["date","kwh"]
Xtr, ytr = tr.drop(columns=drop), tr.kwh
Xva, yva = va.drop(columns=drop), va.kwh
Xte, yte = te.drop(columns=drop), te.kwh
Xf,  yf  = pd.concat([Xtr,Xva]), pd.concat([ytr,yva])

print(len(tr), len(va), len(te), "| test from", te.date.min().date())

In [ ]:
def mape(y, p):
    y, p = np.asarray(y,float), np.asarray(p,float)
    m = y != 0                      # undefined at 0
    return np.abs((y[m]-p[m])/y[m]).mean()*100

def row(name, p):
    return {"model": name,
            "MAE": mean_absolute_error(yte, p),
            "RMSE": mean_squared_error(yte, p)**0.5,
            "MAPE": mape(yte, p)}

out = [row("naive (lag7)", te.lag_7)]

out.append(row("ridge", Ridge(alpha=1).fit(Xf, yf).predict(Xte)))

rf = RandomForestRegressor(n_estimators=400, min_samples_leaf=2,
                           random_state=SEED, n_jobs=-1).fit(Xf, yf)
out.append(row("random forest", rf.predict(Xte)))

# find n_estimators on val, then refit on train+val
tmp = XGBRegressor(n_estimators=2000, learning_rate=.05, max_depth=4,
                   subsample=.8, colsample_bytree=.8, random_state=SEED,
                   early_stopping_rounds=50, eval_metric="mae")
tmp.fit(Xtr, ytr, eval_set=[(Xva,yva)], verbose=False)
best = tmp.best_iteration + 1

xgb = XGBRegressor(n_estimators=best, learning_rate=.05, max_depth=4,
                   subsample=.8, colsample_bytree=.8, random_state=SEED).fit(Xf, yf)
out.append(row(f"xgboost ({best} trees)", xgb.predict(Xte)))

res = pd.DataFrame(out).round(3)
res["vs_naive_%"] = ((res.MAE[0] - res.MAE) / res.MAE[0] * 100).round(1)
res = res.sort_values("MAE").reset_index(drop=True)
res.to_csv("results/comparison_table.csv", index=False)
res

In [ ]:
imp = pd.Series(xgb.feature_importances_, index=Xf.columns).nlargest(15)
imp.to_csv("results/xgb_feature_importance.csv", header=["importance"])

imp.sort_values().plot.barh(figsize=(6,4))
plt.title("xgboost importance")
plt.savefig("results/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
p = xgb.predict(Xte)
plt.figure(figsize=(13,3.5))
plt.plot(te.date, yte.values, lw=.8, label="actual")
plt.plot(te.date, p, lw=.8, label="predicted")
plt.legend(); plt.title("test period"); plt.show()

### Save

In [ ]:
for f in ["data/household_daily.csv", "data/manifest.json",
          "results/comparison_table.csv", "results/xgb_feature_importance.csv",
          "results/feature_importance.png"]:
    files.download(f)

Next: LSTM / GRU. Needs the data as sliding windows instead of lag columns,
so putting that in a separate notebook.